# Desafio prático: Análise Financeira com Python
**Aluno:** Wesley Niederle Sehn

In [16]:
import csv
import json
from datetime import datetime

# Constante solicitada no requisito de transações suspeitas
LIMITE_SUSPEITO = 10000.00

## 1. Leitura e Validação dos Dados
Funções responsáveis por ler o arquivo CSV e descartar linhas com dados inválidos, vazios ou corrompidos.

In [17]:
def ler_transacoes(caminho_arquivo):
    transacoes_validas = []
    total_lidas = 0
    total_validas = 0
    total_invalidas = 0

    try:
        with open(caminho_arquivo, mode='r', encoding='utf-8') as arquivo:
            leitor = csv.DictReader(arquivo)

            for linha in leitor:
                total_lidas += 1
                transacao_limpa = validar_transacao(linha)

                if transacao_limpa is not None:
                    transacoes_validas.append(transacao_limpa)
                    total_validas += 1
                else:
                    total_invalidas += 1

        print(f"Total de linhas lidas: {total_lidas}")
        print(f"Linhas válidas: {total_validas}")
        print(f"Linhas inválidas: {total_invalidas}")
        print("-" * 30)

        return transacoes_validas, total_invalidas, total_validas

    except FileNotFoundError:
        print(f"Erro: O arquivo '{caminho_arquivo}' não foi encontrado.")
        return [], 0, 0

def validar_transacao(linha):
    # Valida ID
    try:
        id_transacao = int(linha['id'])
    except (ValueError, KeyError):
        return None

    # Valida Cliente
    cliente_id = linha.get('cliente_id', '').strip()
    if not cliente_id:
        return None

    # Valida Data
    try:
        data_transacao = datetime.strptime(linha['data'], "%Y-%m-%d")
    except (ValueError, KeyError):
        return None

    # Valida Tipo
    tipo = linha.get('tipo', '').strip().lower()
    if tipo not in ['credito', 'debito']:
        return None

    # Valida Valor
    try:
        valor_transacao = float(linha['valor'])
        if valor_transacao <= 0:
            return None
    except (ValueError, TypeError, KeyError):
        return None

    return {
        'id': id_transacao,
        'data': data_transacao,
        'cliente_id': cliente_id,
        'tipo': tipo,
        'valor': valor_transacao,
        'descricao': linha.get('descricao', ''),
        'categoria': linha.get('categoria', '')
    }

## 2. Processamento e Métricas
Função que agrupa os dados válidos e calcula saldos, médias e identifica transações suspeitas.

In [18]:
def gerar_relatorio(transacoes_validas, total_invalidas, total_validas):
    if not transacoes_validas:
        return {}

    datas = [t['data'] for t in transacoes_validas]
    data_inicio = min(datas) # Data mais antiga
    data_fim = max(datas) # Data mais recente
    dias_analisados = (data_fim - data_inicio).days # Diferença em dias

    resumo_mensal = {}
    transacoes_suspeitas = []

    for t in transacoes_validas:
        mes = t['data'].strftime("%Y-%m")
        valor = t['valor']

        # Estrutura inicial
        if mes not in resumo_mensal:
            resumo_mensal[mes] = {
                "quantidade": 0,
                "total_credito": 0.0,
                "total_debito": 0.0,
                "maior_valor": 0.0,
                "menor_valor": float('inf')
            }

        # Atualiza quantidade
        resumo_mensal[mes]["quantidade"] += 1

        # Atualiza crédito ou débito
        if t['tipo'] == 'credito':
            resumo_mensal[mes]["total_credito"] += valor
        else:
            resumo_mensal[mes]["total_debito"] += valor

        # Verifica maior e menor valor do mês
        if valor > resumo_mensal[mes]["maior_valor"]:
            resumo_mensal[mes]["maior_valor"] = valor
        if valor < resumo_mensal[mes]["menor_valor"]:
            resumo_mensal[mes]["menor_valor"] = valor

        # Verifica se a transação é suspeita
        if valor > LIMITE_SUSPEITO:
            transacoes_suspeitas.append({
                "id": t['id'],
                "cliente_id": t['cliente_id'],
                "data": t['data'].strftime("%Y-%m-%d"),
                "valor": valor
            })

    # Calcular saldo e média de cada mês
    for mes, dados in resumo_mensal.items():
        dados["saldo"] = dados["total_credito"] - dados["total_debito"]
        dados["media"] = (dados["total_credito"] + dados["total_debito"]) / dados["quantidade"]

    # Monta relatório mensal
    relatorio_final = {
        "gerado_em": datetime.now().strftime("%Y-%m-%d"),
        "total_transacoes_validas": total_validas,
        "total_transacoes_invalidas": total_invalidas,
        "periodo_analisado": {
            "inicio": data_inicio.strftime("%Y-%m-%d"),
            "fim": data_fim.strftime("%Y-%m-%d"),
            "dias": dias_analisados
        },
        "resumo_mensal": resumo_mensal,
        "transacoes_suspeitas": transacoes_suspeitas
    }

    return relatorio_final

## 3. Saída e Exportação
Funções para exibir os resultados formatados no terminal e salvar o relatório final em JSON.

In [19]:
def exibir_relatorio(dados_relatorio):
    if not dados_relatorio:
        print("Nenhum dado válido para gerar o relatório.")
        return

    def formatar_moeda(valor):
        return f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

    print("\n" + "="*5 + " RESUMO GERAL " + "="*5)
    inicio = dados_relatorio['periodo_analisado']['inicio']
    fim = dados_relatorio['periodo_analisado']['fim']
    print(f"Período analisado: {inicio} → {fim}")
    print(f"Total de transações válidas: {dados_relatorio['total_transacoes_validas']}")
    print(f"Total de transações inválidas: {dados_relatorio['total_transacoes_invalidas']}\n")

    print("="*5 + " RELATÓRIO MENSAL " + "="*5)
    for mes, dados in dados_relatorio['resumo_mensal'].items():
        print(f"Mês: {mes}")
        print(f"  Transações: {dados['quantidade']}")
        print(f"  Total crédito: {formatar_moeda(dados['total_credito'])}")
        print(f"  Total débito:  {formatar_moeda(dados['total_debito'])}")
        print(f"  Saldo:         {formatar_moeda(dados['saldo'])}")
        print(f"  Média:         {formatar_moeda(dados['media'])}")
        print(f"  Maior valor:   {formatar_moeda(dados['maior_valor'])}")
        print(f"  Menor valor:   {formatar_moeda(dados['menor_valor'])}\n")

    print("="*5 + " TRANSAÇÕES SUSPEITAS " + "="*5)
    suspeitas = dados_relatorio['transacoes_suspeitas']
    if not suspeitas:
        print("Nenhuma transação suspeita encontrada.\n")
    else:
        for s in suspeitas:
            print(f"ID: {s['id']} | Cliente: {s['cliente_id']} | Data: {s['data']} | Valor: {formatar_moeda(s['valor'])}")
        print()

def salvar_json(dados_relatorio, nome_arquivo="relatorio.json"):
    if not dados_relatorio:
        return

    with open(nome_arquivo, 'w', encoding='utf-8') as arquivo:
        json.dump(dados_relatorio, arquivo, ensure_ascii=False, indent=2)

    print(f"Arquivo '{nome_arquivo}' gerado com sucesso!")

## 4. Execução Principal
Célula responsável por orquestrar todas as funções acima e gerar o resultado final.

In [20]:

# 1. Verifica e valida os dados do CSV
transacoes_validas, total_invalidas, total_validas = ler_transacoes('transacoes.csv')

# 2. Gera o relatório
dados_do_relatorio = gerar_relatorio(transacoes_validas, total_invalidas, total_validas)

# 3. Exibe os dados na tela
exibir_relatorio(dados_do_relatorio)

# 4. Salva o JSON final
salvar_json(dados_do_relatorio)

Total de linhas lidas: 20
Linhas válidas: 15
Linhas inválidas: 5
------------------------------

===== RESUMO GERAL =====
Período analisado: 2026-01-05 → 2026-04-02
Total de transações válidas: 15
Total de transações inválidas: 5

===== RELATÓRIO MENSAL =====
Mês: 2026-01
  Transações: 5
  Total crédito: R$ 16.000,00
  Total débito:  R$ 751,25
  Saldo:         R$ 15.248,75
  Média:         R$ 3.350,25
  Maior valor:   R$ 12.500,00
  Menor valor:   R$ 120,00

Mês: 2026-02
  Transações: 5
  Total crédito: R$ 19.200,00
  Total débito:  R$ 1.269,90
  Saldo:         R$ 17.930,10
  Média:         R$ 4.093,98
  Maior valor:   R$ 15.000,00
  Menor valor:   R$ 99,90

Mês: 2026-03
  Transações: 4
  Total crédito: R$ 8.700,00
  Total débito:  R$ 2.040,00
  Saldo:         R$ 6.660,00
  Média:         R$ 2.685,00
  Maior valor:   R$ 5.200,00
  Menor valor:   R$ 240,00

Mês: 2026-04
  Transações: 1
  Total crédito: R$ 0,00
  Total débito:  R$ 75,50
  Saldo:         R$ -75,50
  Média:         R$ 75,5